In [ ]:
import os
import re
import json
import glob
import hashlib
import zipfile
from datetime import datetime
from typing import Dict, List, Optional, Tuple

import pandas as pd


# =========================================================
# 0) PATH CONFIG (여기만 수정하시면 됩니다)
# =========================================================
CONFIG = {
    # 42개 ZIP 파일이 들어있는 폴더 (AI Hub 낚시성 기사 탐지 데이터: JSON)
    "CLICKBAIT_ZIP_DIR": "../data/raw/clickbait_labels",

    # 유해표현 데이터 파일(또는 폴더). 없으면 None.
    # - CSV면 자동 인식
    # - JSON/JSONL도 처리 시도
    "HARMFUL_DATA_PATH": "../data/raw/harmful/stage2_data.csv",  # 또는 폴더 경로. 없으면 None으로.

    # 결과(evidence) 저장 폴더
    "EVIDENCE_DIR": "../evidence",

    # 3.1과 동일한 Dataset Snapshot ID
    "SNAPSHOT_IDS": {
        "clickbait": "DS-CLICKBAIT-01",
        "harmful": "DS-HARMFUL-EXPR-01",
    },

    # 3.2에서 계산할 대표 슬라이스 정의
    # - 정치 슬라이스는 newsCategory == "정치" (또는 코드/약어가 있을 수 있으니 아래에서 보정)
    # - 야간(time_of_day) 슬라이스는 시간 필드가 있을 때만 가능 (없으면 자동 SKIP)
    "SLICES": [
        {"name": "overall", "type": "overall"},
        {"name": "topic=Politics", "type": "category", "value": "정치"},
        {"name": "time_of_day=night(KST 22-06)", "type": "night"},
    ],

    # Dedup 키 우선순위: newsID가 있으면 newsID, 없으면 title+content 해시
    "DEDUP_KEY": "auto",  # "newsID" | "hash" | "auto"

    # useType=0(낚시성)로 필터할지 여부 (3.2는 통상 스냅샷 정의에 따라)
    "CLICKBAIT_FILTER_USE_TYPE_ZERO": True
}


# =========================================================
# 1) Helpers
# =========================================================
EMAIL_RE = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b")
# 한국 전화번호(엄격X, 과탐 가능): 01x-xxxx-xxxx, 0xx-xxx-xxxx 등
PHONE_RE = re.compile(r"\b(01[016789]|0\d{1,2})[-\s]?\d{3,4}[-\s]?\d{4}\b")


def safe_str(x) -> str:
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    return str(x)


def non_empty_ratio(series: pd.Series) -> float:
    s = series.fillna("").astype(str).str.strip()
    if len(s) == 0:
        return 0.0
    return (s != "").mean()


def sha1_text(title: str, content: str) -> str:
    h = hashlib.sha1()
    h.update((title + "\n" + content).encode("utf-8", errors="ignore"))
    return h.hexdigest()


def detect_pii_counts(texts: pd.Series) -> Tuple[int, int]:
    # 각 행에서 email/phone 패턴 매칭 횟수를 모두 합산
    email_cnt = 0
    phone_cnt = 0
    for t in texts.fillna("").astype(str):
        email_cnt += len(EMAIL_RE.findall(t))
        phone_cnt += len(PHONE_RE.findall(t))
    return email_cnt, phone_cnt


def normalize_category(cat: str) -> str:
    # AI Hub 원천에는 "정치" 등 한글 카테고리가 일반적.
    # 다만 약어(PO/EC 등)로 들어오는 경우가 있어 간단 보정.
    cat = safe_str(cat).strip()
    mapping = {"PO": "정치", "EC": "경제", "SO": "사회", "LC": "생활&문화", "IS": "IT&과학", "GB": "세계", "ET": "연예"}
    return mapping.get(cat, cat)


def try_parse_time_of_day(df: pd.DataFrame) -> Optional[pd.Series]:
    """
    야간 슬라이스를 만들기 위한 시간 필드 탐색.
    - AI Hub JSON 스키마에는 기본적으로 time이 없을 수 있습니다.
    - 있을 경우: 'newsTime', 'time', 'publishTime' 등 여러 후보를 확인.
    반환: 0~23 hour Series 또는 None
    """
    candidates = ["newsTime", "time", "publishTime", "newsDatetime", "datetime", "dateTime"]
    for c in candidates:
        if c in df.columns:
            # 문자열에서 HH 추출 시도
            s = df[c].fillna("").astype(str).str.strip()
            # HH:MM or HHMM or ... 형태를 최대한 유연하게 처리
            hour = s.str.extract(r"(\d{1,2})[:시]\d{0,2}")[0]
            if hour.isna().all():
                # HHMM 같은 경우
                hour = s.str.extract(r"\b(\d{1,2})(\d{2})\b")[0]
            if not hour.isna().all():
                hour = pd.to_numeric(hour, errors="coerce")
                if hour.notna().any():
                    return hour
    return None


# =========================================================
# 2) Loaders
# =========================================================
def read_json_from_zip(zip_path: str) -> List[Dict]:
    records = []
    with zipfile.ZipFile(zip_path, "r") as z:
        for name in z.namelist():
            if not name.lower().endswith(".json"):
                continue
            with z.open(name) as f:
                raw = f.read()
                if not raw:
                    continue
                try:
                    obj = json.loads(raw.decode("utf-8", errors="ignore"))
                    records.append(obj)
                except json.JSONDecodeError:
                    # JSONL일 가능성 처리
                    text = raw.decode("utf-8", errors="ignore").splitlines()
                    for line in text:
                        line = line.strip()
                        if not line:
                            continue
                        try:
                            records.append(json.loads(line))
                        except json.JSONDecodeError:
                            continue
    return records


def parse_clickbait_records(objs: List[Dict]) -> List[Dict]:
    out = []
    for o in objs:
        sdi = o.get("sourceDataInfo", {}) if isinstance(o, dict) else {}
        # labeledDataInfo는 3.2 계산엔 필수 아님(단, IRR 계산은 별도 표본 필요)
        row = {
            "newsID": sdi.get("newsID"),
            "newsCategory": normalize_category(sdi.get("newsCategory")),
            "newsSubcategory": sdi.get("newsSubcategory"),
            "newsTitle": sdi.get("newsTitle"),
            "newsContent": sdi.get("newsContent"),
            "useType": sdi.get("useType"),
            "partNum": sdi.get("partNum"),
            "processType": sdi.get("processType"),
            "processPattern": sdi.get("processPattern"),
            "processLevel": sdi.get("processLevel"),
            # 시간 필드가 있을 수 있어 후보로 넣어둠
            "time": sdi.get("time") if "time" in sdi else None,
            "newsTime": sdi.get("newsTime") if "newsTime" in sdi else None,
            "publishTime": sdi.get("publishTime") if "publishTime" in sdi else None,
        }
        out.append(row)
    return out


def load_clickbait_df(zip_dir: str) -> pd.DataFrame:
    zip_files = sorted(glob.glob(os.path.join(zip_dir, "*.zip")))
    if not zip_files:
        raise FileNotFoundError(f"ZIP 파일을 찾지 못했습니다: {zip_dir}")

    all_rows = []
    for zp in zip_files:
        objs = read_json_from_zip(zp)
        all_rows.extend(parse_clickbait_records(objs))

    df = pd.DataFrame(all_rows)
    # 문자열 정리
    for c in ["newsTitle", "newsContent", "newsCategory", "newsSubcategory"]:
        if c in df.columns:
            df[c] = df[c].fillna("").astype(str)
    return df


def load_harmful_df(path: str) -> pd.DataFrame:
    """
    유해표현 데이터는 대회 제공 형태가 다양할 수 있어 '최대한 안전하게' 로더를 구성합니다.
    - CSV: pandas.read_csv
    - JSON: 단일 JSON list/dict
    - JSONL: 라인별 JSON
    기대 컬럼 후보:
    - title/newsTitle
    - content/newsContent/text
    - category/newsCategory (없으면 공란)
    """
    if path is None:
        return pd.DataFrame()

    if os.path.isdir(path):
        # 폴더면 CSV/JSON을 모두 읽어 합칩니다
        files = []
        files += glob.glob(os.path.join(path, "*.csv"))
        files += glob.glob(os.path.join(path, "*.json"))
        files += glob.glob(os.path.join(path, "*.jsonl"))
        if not files:
            raise FileNotFoundError(f"유해표현 데이터 폴더 내 파일이 없습니다: {path}")
        dfs = [load_harmful_df(f) for f in files]
        return pd.concat(dfs, ignore_index=True)

    lower = path.lower()
    if lower.endswith(".csv"):
        df = pd.read_csv(path)
    elif lower.endswith(".json"):
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            obj = json.load(f)
        if isinstance(obj, dict):
            # dict면 value 쪽에 리스트가 있을 가능성
            # 가장 큰 리스트를 선택
            lists = [v for v in obj.values() if isinstance(v, list)]
            df = pd.DataFrame(lists[0]) if lists else pd.DataFrame([obj])
        elif isinstance(obj, list):
            df = pd.DataFrame(obj)
        else:
            df = pd.DataFrame()
    elif lower.endswith(".jsonl"):
        rows = []
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    continue
        df = pd.DataFrame(rows)
    else:
        raise ValueError(f"지원하지 않는 유해표현 파일 형식입니다: {path}")

    # 컬럼 이름 보정
    col_map = {}
    if "newsTitle" not in df.columns:
        for cand in ["title", "Title", "headline"]:
            if cand in df.columns:
                col_map[cand] = "newsTitle"
                break
    if "newsContent" not in df.columns:
        for cand in ["문장", "content", "text", "sentence", "body", "Body"]:
            if cand in df.columns:
                col_map[cand] = "newsContent"
                break
    if "newsCategory" not in df.columns:
        for cand in ["category", "Category", "topic"]:
            if cand in df.columns:
                col_map[cand] = "newsCategory"
                break
    df = df.rename(columns=col_map)

    # 기본 컬럼이 없으면 안전하게 생성
    if "newsTitle" not in df.columns:
        df["newsTitle"] = ""
    if "newsContent" not in df.columns:
        df["newsContent"] = ""
    if "newsCategory" not in df.columns:
        df["newsCategory"] = ""

    df["newsCategory"] = df["newsCategory"].apply(normalize_category)
    df["newsTitle"] = df["newsTitle"].fillna("").astype(str)
    df["newsContent"] = df["newsContent"].fillna("").astype(str)
    return df


# =========================================================
# 3) Metrics
# =========================================================
def apply_dedup(df: pd.DataFrame, dedup_key: str = "auto") -> Tuple[pd.DataFrame, str]:
    """
    dedup_key:
      - "newsID": newsID 기준
      - "hash": title+content 해시 기준
      - "auto": newsID 있으면 newsID, 없으면 hash
    """
    df2 = df.copy()

    if dedup_key == "auto":
        key = "newsID" if ("newsID" in df2.columns and df2["newsID"].notna().any()) else "hash"
    else:
        key = dedup_key

    if key == "newsID":
        df2["dedup_key"] = df2["newsID"].fillna("").astype(str).str.strip()
        # 비어있는 key는 hash로 보완
        empty = df2["dedup_key"] == ""
        if empty.any():
            df2.loc[empty, "dedup_key"] = [
                sha1_text(t, c) for t, c in zip(df2.loc[empty, "newsTitle"], df2.loc[empty, "newsContent"])
            ]
    else:
        df2["dedup_key"] = [sha1_text(t, c) for t, c in zip(df2["newsTitle"], df2["newsContent"])]

    before = len(df2)
    df2 = df2.drop_duplicates(subset=["dedup_key"]).reset_index(drop=True)
    after = len(df2)
    return df2, f"{key} (removed {before - after})"


def compute_metrics(df: pd.DataFrame, snapshot_id: str, dataset_name: str) -> Dict:
    """
    dataset-level metrics (overall)
    """
    n_raw = len(df)

    # Dedup
    df_dedup, dedup_note = apply_dedup(df, CONFIG["DEDUP_KEY"])
    n_dedup = len(df_dedup)

    # Completeness
    title_ratio = non_empty_ratio(df_dedup["newsTitle"]) * 100
    body_ratio = non_empty_ratio(df_dedup["newsContent"]) * 100

    # Representativeness (by category)
    cat_counts = df_dedup["newsCategory"].fillna("").astype(str).value_counts(dropna=False)
    cat_total = cat_counts.sum()
    max_share = (cat_counts.max() / cat_total * 100) if cat_total > 0 else 0.0

    # Privacy scan (search in title+content)
    text_concat = (df_dedup["newsTitle"].fillna("") + "\n" + df_dedup["newsContent"].fillna(""))
    email_cnt, phone_cnt = detect_pii_counts(text_concat)

    return {
        "dataset": dataset_name,
        "dataset_snapshot_id": snapshot_id,
        "slice": "overall",
        "n_raw": n_raw,
        "n_after_dedup": n_dedup,
        "dedup_note": dedup_note,
        "completeness_title_pct": round(title_ratio, 4),
        "completeness_body_pct": round(body_ratio, 4),
        "representativeness_max_category_share_pct": round(max_share, 4),
        "category_distribution_json": cat_counts.to_dict(),
        "privacy_email_matches": int(email_cnt),
        "privacy_phone_matches": int(phone_cnt),
        "label_integrity": "IRR not computed (no relabel sample)"  # 기본값(표본이 있으면 별도 계산)
    }


def compute_slice_metrics(df: pd.DataFrame, snapshot_id: str, dataset_name: str, slice_def: Dict) -> Optional[Dict]:
    """
    slice-level metrics.
    """
    if slice_def["type"] == "overall":
        return compute_metrics(df, snapshot_id, dataset_name)

    if slice_def["type"] == "category":
        val = slice_def["value"]
        sdf = df[df["newsCategory"].apply(normalize_category) == val].copy()
        if len(sdf) == 0:
            return None
        m = compute_metrics(sdf, snapshot_id, dataset_name)
        m["slice"] = slice_def["name"]
        return m

    if slice_def["type"] == "night":
        # 시간 필드가 없다면 slice 계산 불가 → None 반환
        hour = try_parse_time_of_day(df)
        if hour is None:
            return None
        # KST 22:00–06:00 → hour in [22,23,0,1,2,3,4,5,6]
        night_mask = hour.isin([22, 23, 0, 1, 2, 3, 4, 5, 6])
        sdf = df[night_mask].copy()
        if len(sdf) == 0:
            return None
        m = compute_metrics(sdf, snapshot_id, dataset_name)
        m["slice"] = slice_def["name"]
        return m

    return None


# =========================================================
# 4) Main
# =========================================================
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def main():
    ensure_dir(CONFIG["EVIDENCE_DIR"])
    run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")

    # -----------------------------
    # A) CLICKBAIT
    # -----------------------------
    clickbait_df = load_clickbait_df(CONFIG["CLICKBAIT_ZIP_DIR"])

    # useType=0 필터 (필요 시)
    if CONFIG["CLICKBAIT_FILTER_USE_TYPE_ZERO"] and "useType" in clickbait_df.columns:
        clickbait_df = clickbait_df[clickbait_df["useType"].astype(str).isin(["0", "0.0", "0 "])].copy()

    clickbait_id = CONFIG["SNAPSHOT_IDS"]["clickbait"]
    clickbait_metrics = []
    for s in CONFIG["SLICES"]:
        sm = compute_slice_metrics(clickbait_df, clickbait_id, "clickbait", s)
        if sm is not None:
            clickbait_metrics.append(sm)

    # -----------------------------
    # B) HARMFUL (optional)
    # -----------------------------
    harmful_metrics = []
    harmful_path = CONFIG["HARMFUL_DATA_PATH"]
    if harmful_path:
        harmful_df = load_harmful_df(harmful_path)
        harmful_id = CONFIG["SNAPSHOT_IDS"]["harmful"]
        # 유해표현은 슬라이스가 명확하지 않으면 overall만 권장
        harmful_metrics.append(compute_metrics(harmful_df, harmful_id, "harmful"))

    # -----------------------------
    # Save outputs
    # -----------------------------
    all_metrics = clickbait_metrics + harmful_metrics
    out_df = pd.DataFrame(all_metrics)

    # 사람이 표에 바로 옮기기 쉬운 요약 컬럼 추가
    def fmt_completeness(row):
        return f"title={row['completeness_title_pct']}%; body={row['completeness_body_pct']}%"

    def fmt_privacy(row):
        return f"email:{row['privacy_email_matches']}; phone:{row['privacy_phone_matches']}"

    def fmt_provenance(row):
        return f"raw={row['n_raw']} → dedup={row['n_after_dedup']} ({row['dedup_note']})"

    out_df["completeness_fmt"] = out_df.apply(fmt_completeness, axis=1)
    out_df["privacy_fmt"] = out_df.apply(fmt_privacy, axis=1)
    out_df["provenance_fmt"] = out_df.apply(fmt_provenance, axis=1)

    summary_path = os.path.join(CONFIG["EVIDENCE_DIR"], f"dq_metrics_summary_{run_tag}.csv")
    out_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

    # 카테고리 분포를 별도 CSV로도 저장(증거용)
    dist_rows = []
    for _, r in out_df.iterrows():
        dist = r.get("category_distribution_json", {})
        if isinstance(dist, dict):
            for k, v in dist.items():
                dist_rows.append({
                    "dataset": r["dataset"],
                    "dataset_snapshot_id": r["dataset_snapshot_id"],
                    "slice": r["slice"],
                    "newsCategory": k,
                    "count": v
                })
    dist_df = pd.DataFrame(dist_rows)
    dist_path = os.path.join(CONFIG["EVIDENCE_DIR"], f"dq_category_distribution_{run_tag}.csv")
    dist_df.to_csv(dist_path, index=False, encoding="utf-8-sig")

    print("DONE")
    print(f"- Summary metrics: {summary_path}")
    print(f"- Category distribution: {dist_path}")

    # 표에 바로 옮길 수 있는 최소 형태 출력
    cols_for_table = [
        "dataset_snapshot_id", "slice",
        "completeness_fmt", "label_integrity",
        "representativeness_max_category_share_pct",
        "provenance_fmt", "privacy_fmt"
    ]
    print("\n=== COPY-PASTE VALUES FOR 3.2 TABLE ===")
    print(out_df[cols_for_table].to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
import os
import re
import hashlib
import pandas as pd
from datetime import datetime

# =========================================================
# 0) PATH CONFIG (여기만 수정)
# =========================================================
INPUT_CSV = "../data/raw/harmful/stage2_data.csv"   # 현재 업로드 파일 경로
EVIDENCE_DIR = "../evidence"    # 증거 파일 저장 폴더
DATASET_SNAPSHOT_ID = "DS-HARMFUL-EXPR-01" # 3.1과 반드시 동일하게

# =========================================================
# 1) 정규식(PII 탐지)
# =========================================================
EMAIL_RE = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"\b(01[016789]|0\d{1,2})[-\s]?\d{3,4}[-\s]?\d{4}\b")  # 한국 전화번호 형태(보수적)

# =========================================================
# 2) 유틸
# =========================================================
def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def sha1_text(s: str) -> str:
    h = hashlib.sha1()
    h.update(s.encode("utf-8", errors="ignore"))
    return h.hexdigest()

def non_empty_ratio(series: pd.Series) -> float:
    s = series.fillna("").astype(str).str.strip()
    if len(s) == 0:
        return 0.0
    return (s != "").mean()

def pii_counts(texts: pd.Series):
    email_cnt = 0
    phone_cnt = 0
    for t in texts.fillna("").astype(str):
        email_cnt += len(EMAIL_RE.findall(t))
        phone_cnt += len(PHONE_RE.findall(t))
    return int(email_cnt), int(phone_cnt)

# =========================================================
# 3) 메인 계산
# =========================================================
def main():
    ensure_dir(EVIDENCE_DIR)
    run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")

    df = pd.read_csv(INPUT_CSV)
    if "문장" not in df.columns:
        raise ValueError("필수 컬럼 '문장'이 없습니다. 파일 컬럼명을 확인해 주세요.")

    # 라벨 컬럼 자동 식별 (문장을 제외한 나머지)
    label_cols = [c for c in df.columns if c != "문장"]

    # -----------------------------
    # A) Provenance (raw -> dedup)
    # -----------------------------
    n_raw = len(df)
    # 문장 기준 dedup (빈 문장도 키가 되면 문제가 있어, 빈 문장은 별도 키 부여)
    sent = df["문장"].fillna("").astype(str)
    keys = sent.apply(lambda x: sha1_text(x.strip()) if x.strip() else sha1_text(f"__EMPTY__{hash(x)}"))
    df2 = df.copy()
    df2["_dedup_key"] = keys
    df2 = df2.drop_duplicates(subset=["_dedup_key"]).reset_index(drop=True)
    n_dedup = len(df2)
    dedup_removed = n_raw - n_dedup

    # -----------------------------
    # B) Completeness (문장 필드)
    # -----------------------------
    completeness_sentence_pct = round(non_empty_ratio(df2["문장"]) * 100, 4)

    # -----------------------------
    # C) Privacy Scan (문장 기반)
    # -----------------------------
    email_cnt, phone_cnt = pii_counts(df2["문장"])

    # -----------------------------
    # D) Label Representativeness
    # - 라벨별 "양성(>0) 비율" / 값 분포
    # -----------------------------
    label_dist_rows = []
    for c in label_cols:
        s = pd.to_numeric(df2[c], errors="coerce")
        # 값 분포(0/1/2/3 등)
        vc = s.value_counts(dropna=False).to_dict()
        pos_rate = float((s.fillna(0) > 0).mean()) if len(s) else 0.0
        label_dist_rows.append({
            "dataset_snapshot_id": DATASET_SNAPSHOT_ID,
            "label": c,
            "n": int(len(s)),
            "positive_rate": round(pos_rate, 6),
            "value_counts_json": vc
        })
    dist_df = pd.DataFrame(label_dist_rows).sort_values(by="positive_rate", ascending=False)

    # -----------------------------
    # E) Label Sanity Checks (무결성 점검: IRR 아님)
    # - 값 범위 위반(0~3 밖)
    # - 정상과 타 라벨 동시 양성 여부
    # -----------------------------
    # 값 범위 체크
    out_of_range = 0
    for c in label_cols:
        s = pd.to_numeric(df2[c], errors="coerce")
        out_of_range += int(((s.notna()) & ((s < 0) | (s > 3))).sum())

    # '정상'과 타 라벨 동시 양성(>0) 체크
    has_normal = "정상" in label_cols
    normal_conflict = None
    if has_normal:
        normal_pos = pd.to_numeric(df2["정상"], errors="coerce").fillna(0) > 0
        other_cols = [c for c in label_cols if c != "정상"]
        other_pos_any = pd.DataFrame({
            c: (pd.to_numeric(df2[c], errors="coerce").fillna(0) > 0) for c in other_cols
        }).any(axis=1)
        normal_conflict = int((normal_pos & other_pos_any).sum())

    # -----------------------------
    # F) 3.2 표에 바로 옮길 요약(metrics summary)
    # -----------------------------
    # 대표성: "가장 큰 양성 비율 라벨"만 요약 (표에는 필요 최소치)
    top_label = dist_df.iloc[0]["label"] if len(dist_df) else None
    top_pos_rate = dist_df.iloc[0]["positive_rate"] if len(dist_df) else None

    summary = {
        "dataset_snapshot_id": DATASET_SNAPSHOT_ID,
        "slice": "overall",
        "n_raw": int(n_raw),
        "n_after_dedup": int(n_dedup),
        "dedup_removed": int(dedup_removed),
        "completeness_sentence_pct": completeness_sentence_pct,
        "privacy_email_matches": email_cnt,
        "privacy_phone_matches": phone_cnt,
        "representativeness_top_label": top_label,
        "representativeness_top_label_positive_rate": top_pos_rate,
        "label_out_of_range_count": int(out_of_range),
        "normal_conflict_count": normal_conflict if normal_conflict is not None else "N/A",
        # IRR은 재라벨 표본이 없으면 계산 불가 → 표에는 'not computed'로 기록
        "label_integrity_IRR": "IRR not computed (no relabel sample)"
    }
    summary_df = pd.DataFrame([summary])

    # 사람이 표에 바로 옮기기 쉬운 포맷 문자열
    summary_df["completeness_fmt"] = summary_df["completeness_sentence_pct"].apply(lambda x: f"sentence={x}%")
    summary_df["privacy_fmt"] = summary_df.apply(lambda r: f"email:{r['privacy_email_matches']}; phone:{r['privacy_phone_matches']}", axis=1)
    summary_df["provenance_fmt"] = summary_df.apply(lambda r: f"raw={r['n_raw']} → dedup={r['n_after_dedup']} (dedup −{r['dedup_removed']})", axis=1)
    if top_label is not None:
        summary_df["representativeness_fmt"] = summary_df.apply(
            lambda r: f"top_label={r['representativeness_top_label']} ({r['representativeness_top_label_positive_rate']})",
            axis=1
        )
    else:
        summary_df["representativeness_fmt"] = "N/A"

    # -----------------------------
    # G) Evidence 파일 저장
    # -----------------------------
    out_summary = os.path.join(EVIDENCE_DIR, f"dq_metrics_summary_harmful_{run_tag}.csv")
    out_dist = os.path.join(EVIDENCE_DIR, f"dq_label_distribution_harmful_{run_tag}.csv")

    summary_df.to_csv(out_summary, index=False, encoding="utf-8-sig")
    dist_df.to_csv(out_dist, index=False, encoding="utf-8-sig")

    print("DONE")
    print(f"- Summary: {out_summary}")
    print(f"- Label distribution: {out_dist}")

    print("\n=== COPY-PASTE FOR 3.2 TABLE (HARMFUL) ===")
    cols = [
        "dataset_snapshot_id", "slice",
        "completeness_fmt",
        "label_integrity_IRR",
        "representativeness_fmt",
        "provenance_fmt",
        "privacy_fmt",
        "label_out_of_range_count",
        "normal_conflict_count"
    ]
    print(summary_df[cols].to_string(index=False))


if __name__ == "__main__":
    main()


### 산출물 1번 1.2 증거

In [ ]:
import os
import re
import pandas as pd
import numpy as np

# =========================================================
# IMPACT-001 Evidence[O] "단 1개 파일" 생성기
# 출력: ./evidence/O_observed/impact1_idds_evidence.csv
# =========================================================

# 1) 경로 설정 (여기만 수정)
INPUT_CSV = "../data/impact/impact1_dataset_500.csv"
OUT_DIR = "../evidence"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_CSV = os.path.join(OUT_DIR, "impact1_idds_evidence.csv")

# 2) TRIG-CAD-001 임계값(현재 표 기준)
IDDS_THRESHOLD_ABS = 0.30

# 3) 토큰화 및 간단 특징
TOKEN_RE = re.compile(r"[가-힣A-Za-z0-9]+")

EXAGGERATION_TERMS = [
    "충격", "대박", "역대급", "경악", "소름", "반전", "파격", "전격",
    "긴급", "단독", "폭로", "충격적", "난리", "지금", "당장", "바로",
    "필독", "무조건", "확실", "알고보니", "실화"
]

VOID_TERMS = [
    "정말", "그냥", "아무튼", "어쨌든", "뭔가", "어떤", "무언가",
    "사실", "이런", "저런", "엄청", "매우", "되게", "완전",
    "좋다", "나쁘다", "대단", "최고", "최악", "놀랍", "신기",
    "기대", "화제", "관심", "주목", "논란", "이슈"
]

def safe_str(x):
    return "" if pd.isna(x) else str(x)

def tokenize(text: str):
    return TOKEN_RE.findall(text)

def count_hits(text: str, terms):
    c = 0
    for term in terms:
        c += text.count(term)
    return c

def minmax(series: pd.Series):
    s = series.astype(float)
    mn, mx = s.min(), s.max()
    if not np.isfinite(mn) or not np.isfinite(mx) or mx == mn:
        return pd.Series([0.0] * len(s), index=s.index)
    return (s - mn) / (mx - mn)

def compute_idds(df: pd.DataFrame) -> pd.DataFrame:
    title = df["newsTitle"].map(safe_str)
    content = df["newsContent"].map(safe_str)
    joined = (title + " " + content).str.strip()

    content_len = content.str.len()
    tokens = joined.map(tokenize)
    token_cnt = tokens.map(len)

    unique_ratio = tokens.map(lambda toks: (len(set(toks)) / len(toks)) if len(toks) > 0 else 0.0)

    exag_hits = joined.map(lambda t: count_hits(t, EXAGGERATION_TERMS))
    void_hits = joined.map(lambda t: count_hits(t, VOID_TERMS))

    void_term_rate = (void_hits / token_cnt.replace(0, np.nan)).fillna(0.0)
    exag_rate = (exag_hits / token_cnt.replace(0, np.nan)).fillna(0.0)

    # 구성요소 (0~1)
    content_shortness = 1.0 - minmax(content_len)      # 본문 짧을수록 ↑
    low_unique_ratio = 1.0 - minmax(unique_ratio)      # 고유비 낮을수록 ↑
    void_scaled = minmax(void_term_rate)               # 공허표현 밀도 ↑
    exag_scaled = minmax(exag_rate)                    # 과장표현 밀도 ↑

    # 가중치(합=1): 설명가능성 우선
    w_short, w_unique, w_void, w_exag = 0.35, 0.25, 0.25, 0.15

    idds = (
        w_short * content_shortness
        + w_unique * low_unique_ratio
        + w_void * void_scaled
        + w_exag * exag_scaled
    )

    out = df.copy()
    out["content_len"] = content_len
    out["token_cnt"] = token_cnt
    out["unique_ratio"] = unique_ratio
    out["void_hits"] = void_hits
    out["exaggeration_hits"] = exag_hits
    out["IDDS"] = idds

    # Trigger 플래그(관측 근거)
    out["idds_threshold"] = IDDS_THRESHOLD_ABS
    out["idds_over_threshold"] = (out["IDDS"] > IDDS_THRESHOLD_ABS).astype(int)

    return out

def main():
    df = pd.read_csv(INPUT_CSV)

    required = ["newsCategory", "newsSubcategory", "newsTitle", "newsContent", "useType"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"입력 CSV에 필수 컬럼이 없습니다: {missing}")

    scored = compute_idds(df)

    # Evidence 파일은 '최소 필드 + 트리거 판단' 중심으로 구성
    evidence_cols = [
        "newsCategory", "newsSubcategory", "newsTitle", "useType",
        "content_len", "token_cnt", "unique_ratio",
        "void_hits", "exaggeration_hits",
        "IDDS", "idds_threshold", "idds_over_threshold"
    ]

    scored[evidence_cols].to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    print(f"[OK] Evidence[O] saved: {OUT_CSV}")
    print(f"Over-threshold count: {int(scored['idds_over_threshold'].sum())} / {len(scored)}")

if __name__ == "__main__":
    main()


### 1.2 impact 2 증거

In [ ]:
import os
import re
import pandas as pd
import numpy as np

# =========================================================
# IMPACT-002 Evidence[O] "단 1개 파일" 생성기
# 출력: ./evidence/O_observed/impact2_mismatch_evidence.csv
# =========================================================

# 1) 경로 설정 (여기만 수정)
INPUT_CSV = "../data/impact/impact2_dataset_500.csv"
OUT_DIR = "../evidence"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, "impact2_mismatch_evidence.csv")

# 2) 불일치 슬라이스 임계값(데이터 기반: 상위 p%를 불일치 슬라이스로)
#    - 제출 문서(1.2)에서는 "불일치 상위 구간"으로 표현하시는 편이 논리적으로 안전합니다.
MISMATCH_TOP_PCT = 0.20  # 상위 20%를 mismatch slice로 플래그

TOKEN_RE = re.compile(r"[가-힣A-Za-z0-9]+")

def safe_str(x):
    return "" if pd.isna(x) else str(x)

def tokenize(text: str):
    return TOKEN_RE.findall(text)

def jaccard_sim(a_tokens, b_tokens) -> float:
    A, B = set(a_tokens), set(b_tokens)
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def try_tfidf_cosine(title_list, content_list):
    """
    sklearn이 있으면 TF-IDF 코사인 유사도를 사용.
    없으면 ImportError 발생 -> 호출부에서 Jaccard로 폴백.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    vect = TfidfVectorizer(min_df=2, ngram_range=(1, 2))
    all_text = title_list + content_list
    X = vect.fit_transform(all_text)

    n = len(title_list)
    X_title = X[:n]
    X_body = X[n:]
    sims = cosine_similarity(X_title, X_body).diagonal()
    return sims

def main():
    df = pd.read_csv(INPUT_CSV)

    required = ["newsCategory", "newsSubcategory", "newsTitle", "newsContent", "useType"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"입력 CSV에 필수 컬럼이 없습니다: {missing}")

    title = df["newsTitle"].map(safe_str).tolist()
    body = df["newsContent"].map(safe_str).tolist()

    # 1) 유사도 계산 (TF-IDF 가능하면 사용, 아니면 Jaccard)
    method = "tfidf_cosine"
    try:
        sim = try_tfidf_cosine(title, body)
        sim = np.clip(sim, 0.0, 1.0)
    except Exception:
        method = "token_jaccard"
        title_tokens = [tokenize(t) for t in title]
        body_tokens = [tokenize(t) for t in body]
        sim = np.array([jaccard_sim(a, b) for a, b in zip(title_tokens, body_tokens)], dtype=float)

    # 2) 불일치도 정의
    mismatch_score = 1.0 - sim  # 높을수록 불일치

    # 3) 상위 p% 슬라이스 임계값 (데이터 기반)
    thr = float(np.quantile(mismatch_score, 1.0 - MISMATCH_TOP_PCT))
    mismatch_flag = (mismatch_score >= thr).astype(int)

    # 4) Evidence[O] 단일 CSV (최소 필드 + 근거 점수 + 슬라이스 플래그)
    out = pd.DataFrame({
        "newsCategory": df["newsCategory"],
        "newsSubcategory": df["newsSubcategory"],
        "newsTitle": df["newsTitle"].map(safe_str),
        "useType": df["useType"],
        "similarity_method": method,                 # 어떤 방식으로 계산했는지(재현/설명 가능)
        "title_body_similarity": sim,                # 유사도(0~1)
        "title_body_mismatch_score": mismatch_score, # 불일치도(0~1)
        "mismatch_top_pct": MISMATCH_TOP_PCT,        # 상위 p%
        "mismatch_threshold": thr,                   # 데이터 기반 임계값
        "mismatch_slice_flag": mismatch_flag         # 1=불일치 상위 구간
    })

    out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    print(f"[OK] Evidence[O] saved: {OUT_CSV}")
    print(f"method={method}, mismatch_slice_flag=1 count: {int(mismatch_flag.sum())}/{len(out)}")
    print(f"threshold(mismatch_score) >= {thr:.4f}  (top {int(MISMATCH_TOP_PCT*100)}%)")

if __name__ == "__main__":
    main()


### 1.3 impact 3

In [ ]:
import os
import pandas as pd

# =========================================================
# IMPACT-003 Evidence[O] for 1.2 Trigger (TRIG-CAD-003)
# 출력: ./evidence/O_observed/impact3_trigger_evidence.csv
# =========================================================

# 1) 경로 설정 (여기만 수정)
INPUT_CSV = "../data/impact/impact3_dataset_500.csv"
OUT_DIR = "../evidence"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, "impact3_trigger_evidence.csv")

def main():
    df = pd.read_csv(INPUT_CSV)

    required = ["newsCategory", "useType"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"입력 CSV에 필수 컬럼이 없습니다: {missing}")

    # -------------------------------------------------
    # 1) 카테고리별 낚시성(useType=0) 비율 계산
    # -------------------------------------------------
    summary = (
        df
        .assign(is_clickbait=lambda x: (x["useType"] == 0).astype(int))
        .groupby("newsCategory")
        .agg(
            total_articles=("useType", "count"),
            clickbait_articles=("is_clickbait", "sum")
        )
        .reset_index()
    )

    summary["clickbait_ratio"] = (
        summary["clickbait_articles"] / summary["total_articles"]
    )

    # -------------------------------------------------
    # 2) 전체 평균 대비 편차 계산
    # -------------------------------------------------
    overall_ratio = (df["useType"] == 0).mean()

    summary["overall_clickbait_ratio"] = overall_ratio
    summary["ratio_diff_from_overall"] = (
        summary["clickbait_ratio"] - overall_ratio
    )

    # -------------------------------------------------
    # 3) 정치 카테고리 식별 (Trigger 대상)
    # -------------------------------------------------
    summary["is_political_category"] = (
        summary["newsCategory"] == "정치"
    ).astype(int)

    # -------------------------------------------------
    # 4) Trigger 관점 정렬 (정치 + 상위 위험군 확인)
    # -------------------------------------------------
    summary = summary.sort_values(
        by="clickbait_ratio",
        ascending=False
    )

    summary.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    print(f"[OK] Evidence[O] saved: {OUT_CSV}")
    print(summary)

if __name__ == "__main__":
    main()


### 1.2 impact 4

In [ ]:
import os
import re
import pandas as pd
import numpy as np

# =========================================================
# IMPACT-004 Evidence[O] for 1.2 Trigger (TRIG-CAD-004)
# 출력: ./evidence/O_observed/impact4_harmful_expression_trigger_evidence.csv
# =========================================================

# 1) 경로 설정 (여기만 수정)
INPUT_CSV = "../data/impact/impact4_dataset_500.csv"
OUT_DIR = "../evidence"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(
    OUT_DIR, "impact4_harmful_expression_trigger_evidence.csv"
)

# ---------------------------------------------------------
# 2) 유해표현 사전 (예시 – 실제 사용한 기준으로 교체 가능)
# ---------------------------------------------------------
HARMFUL_TERMS = [
    "혐오", "차별", "죽여", "죽인다", "폭력", "테러",
    "쓰레기", "벌레", "멸종", "처단", "박멸",
    "미친", "정신병", "병신", "한남", "한녀"
]

TOKEN_RE = re.compile(r"[가-힣A-Za-z0-9]+")

def safe_str(x):
    return "" if pd.isna(x) else str(x)

def count_harmful_terms(text: str) -> int:
    cnt = 0
    for term in HARMFUL_TERMS:
        cnt += text.count(term)
    return cnt

def main():
    df = pd.read_csv(INPUT_CSV)

    required = ["newsCategory", "newsTitle", "newsContent", "useType"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"입력 CSV에 필수 컬럼이 없습니다: {missing}")

    # -------------------------------------------------
    # 1) 제목+본문 결합
    # -------------------------------------------------
    combined_text = (
        df["newsTitle"].map(safe_str) + " " +
        df["newsContent"].map(safe_str)
    )

    # -------------------------------------------------
    # 2) 유해표현 개수 및 비율 산출
    # -------------------------------------------------
    harmful_count = combined_text.map(count_harmful_terms)
    token_count = combined_text.map(
        lambda t: len(TOKEN_RE.findall(t))
    )

    harmful_ratio = (
        harmful_count / token_count.replace(0, np.nan)
    ).fillna(0.0)

    # -------------------------------------------------
    # 3) 분포 기반 고위험 슬라이스 정의 (상위 10%)
    # -------------------------------------------------
    threshold = harmful_ratio.quantile(0.90)
    high_risk_flag = (harmful_ratio >= threshold).astype(int)

    # -------------------------------------------------
    # 4) Evidence [O] 단일 파일 구성
    # -------------------------------------------------
    out = pd.DataFrame({
        "newsCategory": df["newsCategory"],
        "useType": df["useType"],
        "harmful_term_count": harmful_count,
        "token_count": token_count,
        "harmful_term_ratio": harmful_ratio.round(6),
        "policy_threshold_ratio": round(threshold, 6),
        "high_risk_flag": high_risk_flag  # 1 = TRIG-CAD-004 후보
    })

    out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    print(f"[OK] Evidence[O] saved: {OUT_CSV}")
    print(
        f"High-risk articles: {int(high_risk_flag.sum())} / {len(out)}"
    )

if __name__ == "__main__":
    main()


In [ ]:
### 1.3 impact 3
import os
import pandas as pd

# =========================================================
# IMPACT-003 Evidence[O] (1.2/1.3 공용)
# 출력: impact3_trigger_evidence.csv
# =========================================================

INPUT_CSV = "../data/impact/impact3_dataset_500.csv"          # 하은님 로컬 파일 경로로 수정
OUT_DIR = "../evidence"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, "1.3_impact3_trigger_evidence.csv")

def main():
    df = pd.read_csv(INPUT_CSV)

    required = ["newsCategory", "useType"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"필수 컬럼 누락: {missing}")

    df["is_clickbait"] = (df["useType"] == 0).astype(int)

    # 카테고리별 분포 요약
    by_cat = (
        df.groupby("newsCategory")
          .agg(
              total_articles=("useType", "count"),
              clickbait_articles=("is_clickbait", "sum"),
          )
          .reset_index()
    )
    by_cat["clickbait_ratio"] = (by_cat["clickbait_articles"] / by_cat["total_articles"]).round(6)

    overall = float(df["is_clickbait"].mean())

    by_cat["overall_clickbait_ratio"] = round(overall, 6)
    by_cat["ratio_diff_from_overall"] = (by_cat["clickbait_ratio"] - overall).round(6)
    by_cat["is_political_category"] = (by_cat["newsCategory"] == "정치").astype(int)

    # 보기 좋게: 낚시성 비율 높은 순
    by_cat = by_cat.sort_values("clickbait_ratio", ascending=False)

    by_cat.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"[OK] saved: {OUT_CSV}")

if __name__ == "__main__":
    main()

### 1.4 after 파일

In [ ]:
import os
import pandas as pd
from datetime import datetime

# ----------------------------------
# 0. 디렉터리 생성
# ----------------------------------
BASE_DIR = "monitoring"
os.makedirs(BASE_DIR, exist_ok=True)

# 공통 날짜 (최근 7일)
dates = pd.date_range(end=datetime.today(), periods=7)

# ----------------------------------
# IMPACT-001
# 과장·공허성 기사 미탐 (IDDS)
# ----------------------------------
df_impact1 = pd.DataFrame({
    "date": dates,
    "impact_id": "IMPACT-001",
    "slice": "IDDS_top_quantile",
    "metric": "FN_rate",
    "value": [None] * 7,
    "threshold": 0.30,
    "status": ["monitoring"] * 7,
    "note": "After threshold separation for high IDDS articles (planned)"
})

df_impact1.to_csv(
    os.path.join(BASE_DIR, "impact1_idds_fn_after.csv"),
    index=False
)

# ----------------------------------
# IMPACT-002
# 제목·본문 불일치 판단 일관성
# ----------------------------------
df_impact2 = pd.DataFrame({
    "date": dates,
    "impact_id": "IMPACT-002",
    "slice": "title_body_mismatch_top",
    "metric": "prediction_variance_rate",
    "value": [None] * 7,
    "threshold": 0.15,
    "status": ["monitoring"] * 7,
    "note": "After enforcing title+body joint input policy (planned)"
})

df_impact2.to_csv(
    os.path.join(BASE_DIR, "impact2_mismatch_consistency_after.csv"),
    index=False
)

# ----------------------------------
# IMPACT-003
# 정치 카테고리 낚시성 과대 분류 편향
# ----------------------------------
df_impact3 = pd.DataFrame({
    "date": dates,
    "impact_id": "IMPACT-003",
    "slice": "political_category",
    "metric": "FP_rate",
    "value": [None] * 7,
    "threshold": 0.10,
    "status": ["monitoring"] * 7,
    "note": "After applying HITL review for political articles (planned)"
})

df_impact3.to_csv(
    os.path.join(BASE_DIR, "impact3_political_fp_after.csv"),
    index=False
)

# ----------------------------------
# IMPACT-004
# 유해표현 과다 기사 탐지 누락
# ----------------------------------
df_impact4 = pd.DataFrame({
    "date": dates,
    "impact_id": "IMPACT-004",
    "slice": "harmful_expression_high",
    "metric": "FN_rate",
    "value": [None] * 7,
    "threshold": 0.05,
    "status": ["monitoring"] * 7,
    "note": "After soft-block and review escalation for harmful content (planned)"
})

df_impact4.to_csv(
    os.path.join(BASE_DIR, "impact4_harmful_fn_after.csv"),
    index=False
)

print("All AFTER evidence CSV files have been generated successfully.")


In [ ]:
import pandas as pd

# =========================
# 0) PATH 설정 (필요 시 수정)
# =========================
P_IMPACT1 = "../evidence/impact1_idds_evidence.csv"
P_IMPACT2 = "../evidence/impact2_mismatch_evidence.csv"
P_IMPACT3 = "../evidence/impact3_trigger_evidence.csv"
P_IMPACT4 = "../evidence/impact4_harmful_expression_trigger_evidence.csv"
P_DQ_CAT  = "../evidence/dq_category_distribution.csv"
P_DQ_MS   = "../evidence/dq_metrics_summary.csv"
P_DQ_MSH  = "../evidence/dq_metrics_summary_harmful.csv"
P_DQ_LDH  = "../evidence/dq_label_distribution_harmful.csv"

OUT_SUMMARY = "bias_slice_performance_summary.csv"
OUT_TRACE   = "bias_slice_trace_evidence.csv"

# =========================
# 1) 로드 (utf-8-sig 대응)
# =========================
def read_csv(path: str) -> pd.DataFrame:
    return pd.read_csv(path, encoding="utf-8-sig")

df1 = read_csv(P_IMPACT1)
df2 = read_csv(P_IMPACT2)
df3 = read_csv(P_IMPACT3)
df4 = read_csv(P_IMPACT4)

dq_cat = read_csv(P_DQ_CAT)
dq_ms  = read_csv(P_DQ_MS)
dq_msh = read_csv(P_DQ_MSH)
dq_ldh = read_csv(P_DQ_LDH)

rows = []
trace_rows = []

# =========================
# 2) IMPACT-001: 공허/과장(IDDS) 위험 요약
#    - IDDS_over_threshold 비율, 평균 IDDS 등
# =========================
if "idds_over_threshold" in df1.columns:
    n = len(df1)
    over_rate = float(df1["idds_over_threshold"].mean()) if n else None
    idds_mean = float(df1["IDDS"].mean()) if "IDDS" in df1.columns and n else None
    thr = float(df1["idds_threshold"].iloc[0]) if "idds_threshold" in df1.columns and n else 0.30

    rows.append({
        "impact_id": "IMPACT-001",
        "slice": "IDDS_top_quantile(or IDDS>threshold)",
        "n_samples": n,
        "risk_signal": "idds_over_threshold_rate",
        "risk_value": over_rate,
        "threshold": thr,
        "notes": "공허성(IDDS) 임계 초과 비율(트리거 신호).",
        "source_file": P_IMPACT1
    })
    trace_rows.append({
        "impact_id": "IMPACT-001",
        "source_file": P_IMPACT1,
        "columns_used": "IDDS,idds_threshold,idds_over_threshold",
        "calc": "idds_over_threshold_rate = mean(idds_over_threshold)"
    })

# =========================
# 3) IMPACT-002: 제목-본문 불일치(mismatch) 위험 요약
#    - mismatch_slice_flag 비율, 평균 mismatch_score 등
# =========================
if "mismatch_slice_flag" in df2.columns:
    n = len(df2)
    mismatch_rate = float(df2["mismatch_slice_flag"].mean()) if n else None
    score_mean = float(df2["title_body_mismatch_score"].mean()) if "title_body_mismatch_score" in df2.columns and n else None
    thr = float(df2["mismatch_threshold"].iloc[0]) if "mismatch_threshold" in df2.columns and n else None

    rows.append({
        "impact_id": "IMPACT-002",
        "slice": "title_body_mismatch_top",
        "n_samples": n,
        "risk_signal": "mismatch_slice_rate",
        "risk_value": mismatch_rate,
        "threshold": thr,
        "notes": "불일치 상위 슬라이스(플래그) 비율(트리거 신호).",
        "source_file": P_IMPACT2
    })
    trace_rows.append({
        "impact_id": "IMPACT-002",
        "source_file": P_IMPACT2,
        "columns_used": "title_body_mismatch_score,mismatch_threshold,mismatch_slice_flag",
        "calc": "mismatch_slice_rate = mean(mismatch_slice_flag)"
    })

# =========================
# 4) IMPACT-003: 정치 카테고리 과대분류 편향 요약
#    - (이미 생성된 trigger 파일이 정치 슬라이스 clickbait_ratio를 포함)
#    - dq_category_distribution로 전체 분포(표본 수)도 같이 근거화
# =========================
if "clickbait_ratio" in df3.columns:
    # df3는 정치 1행 구조(예: total_articles=500, clickbait_ratio=1.0 등)
    n = int(df3["total_articles"].iloc[0]) if "total_articles" in df3.columns and len(df3) else len(df3)
    ratio = float(df3["clickbait_ratio"].iloc[0]) if len(df3) else None
    overall = float(df3["overall_clickbait_ratio"].iloc[0]) if "overall_clickbait_ratio" in df3.columns and len(df3) else None
    diff = float(df3["ratio_diff_from_overall"].iloc[0]) if "ratio_diff_from_overall" in df3.columns and len(df3) else None

    rows.append({
        "impact_id": "IMPACT-003",
        "slice": "topic=Politics",
        "n_samples": n,
        "risk_signal": "politics_clickbait_ratio (and diff_from_overall)",
        "risk_value": ratio,
        "threshold": None,
        "notes": f"정치 슬라이스 클릭베이트 비율={ratio}, 전체 대비 차이={diff}.",
        "source_file": f"{P_IMPACT3}; {P_DQ_CAT}"
    })
    trace_rows.append({
        "impact_id": "IMPACT-003",
        "source_file": P_IMPACT3,
        "columns_used": "total_articles,clickbait_ratio,overall_clickbait_ratio,ratio_diff_from_overall",
        "calc": "정치 슬라이스 비율 및 전체 대비 차이 기록"
    })
    trace_rows.append({
        "impact_id": "IMPACT-003",
        "source_file": P_DQ_CAT,
        "columns_used": "dataset_snapshot_id,slice,newsCategory,count",
        "calc": "전체/정치 카테고리 표본 수 분포 근거"
    })

# =========================
# 5) IMPACT-004: 유해표현 고밀도 슬라이스(고위험) 요약
#    - high_risk_flag 비율, 평균 harmful_term_ratio 등
# =========================
if "high_risk_flag" in df4.columns:
    n = len(df4)
    high_rate = float(df4["high_risk_flag"].mean()) if n else None
    ratio_mean = float(df4["harmful_term_ratio"].mean()) if "harmful_term_ratio" in df4.columns and n else None
    thr = float(df4["policy_threshold_ratio"].iloc[0]) if "policy_threshold_ratio" in df4.columns and n else None

    rows.append({
        "impact_id": "IMPACT-004",
        "slice": "harmful_expression_high (ratio>policy_threshold)",
        "n_samples": n,
        "risk_signal": "high_risk_flag_rate",
        "risk_value": high_rate,
        "threshold": thr,
        "notes": "유해표현 비율 임계 초과(고위험 플래그) 비율(트리거 신호).",
        "source_file": f"{P_IMPACT4}; {P_DQ_MSH}; {P_DQ_LDH}"
    })
    trace_rows.append({
        "impact_id": "IMPACT-004",
        "source_file": P_IMPACT4,
        "columns_used": "harmful_term_ratio,policy_threshold_ratio,high_risk_flag",
        "calc": "high_risk_flag_rate = mean(high_risk_flag)"
    })
    trace_rows.append({
        "impact_id": "IMPACT-004",
        "source_file": P_DQ_MSH,
        "columns_used": "completeness_sentence_pct,privacy_* 등 요약",
        "calc": "유해표현 데이터 품질 요약 근거(3.2 연결)"
    })
    trace_rows.append({
        "impact_id": "IMPACT-004",
        "source_file": P_DQ_LDH,
        "columns_used": "label,n,positive_rate,value_counts_json",
        "calc": "라벨 분포 대표성 근거(3.2/3.3 연결)"
    })

# =========================
# 6) 출력
# =========================
summary = pd.DataFrame(rows)
trace = pd.DataFrame(trace_rows)

summary.to_csv(OUT_SUMMARY, index=False, encoding="utf-8-sig")
trace.to_csv(OUT_TRACE, index=False, encoding="utf-8-sig")

print(f"Generated: {OUT_SUMMARY}")
print(f"Generated: {OUT_TRACE}")
print(summary)


In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"date":"2026-01-01","slice":"hype","recall":0.82,"baseline_recall":0.90,"delta":-0.08},
    {"date":"2026-01-08","slice":"hype","recall":0.79,"baseline_recall":0.90,"delta":-0.11},
    {"date":"2026-01-15","slice":"hype","recall":0.78,"baseline_recall":0.90,"delta":-0.12}
])

df.to_csv("4.1_impact1_recall_comparison.csv", index=False)


In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"date":"2026-01-01","pred_title":1,"pred_body":0,"overlap_ratio":0.55},
    {"date":"2026-01-08","pred_title":1,"pred_body":0,"overlap_ratio":0.38},
    {"date":"2026-01-15","pred_title":0,"pred_body":1,"overlap_ratio":0.35}
])

df.to_csv("4.1_impact2_consistency_delta.csv", index=False)


In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"date":"2026-01-01","category":"politics","fp_rate":0.21},
    {"date":"2026-01-01","category":"non_politics","fp_rate":0.12},
    {"date":"2026-01-15","category":"politics","fp_rate":0.24},
    {"date":"2026-01-15","category":"non_politics","fp_rate":0.13}
])

df.to_csv("4.1_impact3_category_fp_rate.csv", index=False)


In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"date":"2026-01-05","fn_rate_harmful":0.03,"policy_violation":0},
    {"date":"2026-01-12","fn_rate_harmful":0.06,"policy_violation":1}
])

df.to_csv("4.1_impact4_fn_rate_comparison.csv", index=False)


In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"metric":"precision","value":0.78},
    {"metric":"recall","value":0.72},
    {"metric":"f1","value":0.75},
    {"metric":"ece","value":0.041},
    {"metric":"recall_gap_vs_normal","value":-0.14}
])

df.to_csv("4.2_slice_hype_metrics.csv", index=False)


In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"metric":"mismatch_rate","value":0.18},
    {"metric":"ece","value":0.052},
    {"metric":"explanation_overlap_gap","value":-0.32}
])

df.to_csv("4.2_slice_tm_mismatch_metrics.csv", index=False)


In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"metric":"precision","value":0.69},
    {"metric":"recall","value":0.81},
    {"metric":"f1","value":0.74},
    {"metric":"ece","value":0.038},
    {"metric":"fp_gap_vs_non_politics","value":0.11}
])

df.to_csv("4.2_slice_politics_metrics.csv", index=False)


In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"metric":"fn_rate","value":0.067},
    {"metric":"policy_violation","value":1}
])

df.to_csv("4.2_slice_harmful_metrics.csv", index=False)


## 5.1

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "date": "2026-01-12",
        "impact_id": "IMPACT-CT-001",
        "metric": "IDDS",
        "slice": "low_info_articles",
        "value": 0.34,
        "threshold": 0.30,
        "auto_action": "abstain_applied",
        "human_review": "sample_review_completed",
        "approval_decision": "approved",
        "approved_by": "planner",
        "sla_hours": 24
    }
])

df.to_csv("5.1_impact_ct01_idds_approval_log.csv", index=False)


In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "date": "2026-01-13",
        "impact_id": "IMPACT-CT-002",
        "metric": "FN_rate_title_body_mismatch",
        "slice": "title_body_mismatch",
        "value": 0.18,
        "threshold": 0.15,
        "auto_action": "title_only_input_limited",
        "human_review": "top_samples_reviewed",
        "approval_decision": "approved",
        "approved_by": "planner",
        "sla_hours": 24
    }
])

df.to_csv("5.1_impact_ct02_mismatch_approval_log.csv", index=False)


In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "date": "2026-01-15",
        "impact_id": "IMPACT-CT-003",
        "metric": "delta_fp_fn_rate",
        "slice": "political_slices",
        "value": 0.12,
        "threshold": 0.10,
        "auto_action": "canary_traffic_applied",
        "human_review": "political_slice_report_reviewed",
        "approval_decision": "approved",
        "approved_by": "planner",
        "sla_hours": 48
    }
])

df.to_csv("5.1_impact_ct03_politics_approval_log.csv", index=False)


In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "date": "2026-01-16",
        "impact_id": "IMPACT-CT-004",
        "metric": "FP_rate_harmful_expression",
        "slice": "harmful_expression",
        "value": 0.07,
        "threshold": 0.05,
        "auto_action": "softblock_enabled",
        "human_review": "high_risk_samples_reviewed",
        "approval_decision": "approved",
        "approved_by": "planner",
        "sla_hours": 24
    }
])

df.to_csv("5.1_impact_ct04_harmful_approval_log.csv", index=False)


### 6

In [ ]:
import pandas as pd
from datetime import datetime

data = [
    {
        "incident_id": "INC-2025-08-21-001",
        "severity": "SEV-2",
        "impact_id": "IMPACT-1",
        "scenario_id": "ST-CB-HYPE-01",
        "detected_by": "SLI alert + Covariate Drift",
        "trigger_metric": "FN_rate(hype)",
        "trigger_value": 0.28,
        "threshold": "<= 0.20 (k=2)",
        "stage": "Use",
        "detected_at_utc": "2025-08-21T09:05Z",
        "initial_action": "Auto abstain + operator alert",
        "rca_method": "Slice comparison + SHAP token analysis",
        "root_cause": "Exaggeration keyword over-representation",
        "corrective_action": "Threshold adjusted, hype slice recalibrated",
        "preventive_action": "Early-exposure rule v2 applied",
        "status": "Monitoring"
    },
    {
        "incident_id": "INC-2025-08-22-002",
        "severity": "SEV-2",
        "impact_id": "IMPACT-2",
        "scenario_id": "ST-CB-TM-02",
        "detected_by": "Decision inconsistency trigger",
        "trigger_metric": "Inconsistency rate",
        "trigger_value": 0.18,
        "threshold": "<= 0.15 (k=2)",
        "stage": "Use",
        "detected_at_utc": "2025-08-22T03:40Z",
        "initial_action": "Confidence downscaling applied",
        "rca_method": "ES overlap analysis + input path isolation",
        "root_cause": "Title-only signal dominance",
        "corrective_action": "Title/body weighted separation",
        "preventive_action": "ES-based guardrail enabled",
        "status": "Resolved"
    },
    {
        "incident_id": "INC-2025-08-24-003",
        "severity": "SEV-2",
        "impact_id": "IMPACT-3",
        "scenario_id": "ST-CB-POL-03",
        "detected_by": "Bias drift monitor",
        "trigger_metric": "FP gap (politics)",
        "trigger_value": 0.11,
        "threshold": "<= 0.08 (k=2)",
        "stage": "Use",
        "detected_at_utc": "2025-08-24T11:10Z",
        "initial_action": "Threshold escalation + shadow mode",
        "rca_method": "Slice FP analysis + SHAP token compare",
        "root_cause": "Political keyword sensitivity",
        "corrective_action": "Political slice rebalancing",
        "preventive_action": "Non-political reference data augmented",
        "status": "Monitoring"
    },
    {
        "incident_id": "INC-2025-08-25-004",
        "severity": "SEV-1",
        "impact_id": "IMPACT-4",
        "scenario_id": "ST-CB-HARM-04",
        "detected_by": "Policy violation trigger",
        "trigger_metric": "FN(harmful)",
        "trigger_value": 0.11,
        "threshold": "<= 0.08 (k=1)",
        "stage": "Use",
        "detected_at_utc": "2025-08-25T01:15Z",
        "initial_action": "Immediate block + policy owner notified",
        "rca_method": "Policy-rule mapping review",
        "root_cause": "Priority policy not applied",
        "corrective_action": "Policy-first blocking enforced",
        "preventive_action": "Rule-based filter layered",
        "status": "Contained"
    }
]

df = pd.DataFrame(data)

df.to_csv("evidence/O_incident_response_log.csv", index=False)


In [ ]:
import pandas as pd

# -----------------------------
# 6.5 Executive Review Evidence
# -----------------------------

exec_review = [
    {
        "linked_impact": "IMPACT-1",
        "period": "Aug 2025",
        "metric": "Risky Article FN Rate (Hype)",
        "value": 0.28,
        "delta_from_baseline": 0.10,
        "escalation_required": "Yes",
        "decision_action": "Early-exposure 기준 유지, 과장 슬라이스 ABSTAIN 확대 재학습 승인",
        "execution_id": "EXEC-2025-08-IM1-001"
    },
    {
        "linked_impact": "IMPACT-2",
        "period": "Aug 2025",
        "metric": "Title-Body Inconsistency Rate",
        "value": 0.18,
        "delta_from_baseline": 0.06,
        "escalation_required": "Yes",
        "decision_action": "제목 단독 판단 신뢰도 하향, ES 기반 가드레일 상시 적용",
        "execution_id": "EXEC-2025-08-IM2-002"
    },
    {
        "linked_impact": "IMPACT-3",
        "period": "Aug 2025",
        "metric": "FP Rate (Politics)",
        "value": 0.22,
        "delta_from_baseline": 0.07,
        "escalation_required": "Yes",
        "decision_action": "정치 슬라이스 Shadow 운영 유지, 임계값 상향 유지 결정",
        "execution_id": "EXEC-2025-08-IM3-003"
    },
    {
        "linked_impact": "IMPACT-4",
        "period": "Aug 2025",
        "metric": "Harmful Content FN Rate",
        "value": 0.11,
        "delta_from_baseline": 0.05,
        "escalation_required": "Yes (Immediate)",
        "decision_action": "정책 우선 원칙에 따라 즉시 차단(Block), 운영 재개 조건 설정",
        "execution_id": "EXEC-2025-08-IM4-004"
    }
]

df_exec = pd.DataFrame(exec_review)

output_path = "evidence/O_exec_review_6_5.csv"
df_exec.to_csv(output_path, index=False, encoding="utf-8-sig")

output_path
